In [7]:
import pandas as pd
import sys
import os

# Get the parent directory of the notebooks folder
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
parent_dir = os.path.dirname(notebook_dir)
sys.path.insert(0, parent_dir)

from src import preprocessing as pre

### Encoding for Dataset1

In [20]:
df = pd.read_csv('../data/processed_data/cybersecurity_attacks_dataset1.csv')


In [21]:
# # OPTIONAL: only if needed (change wrong name in Anomaly Scores_Quartile) before
df['Anomaly Scores_Quartile'] = df['Anomaly Scores_Quartile'].apply(
    lambda x: 'Low (0-24)' if x == 'Low (0-25)'
    else 'Medium (25-49)' if x == 'Medium (25-50)'
    else 'High (50-74)' if x == 'High (50-75)'
    else 'Very High (75-100)'
    )

In [17]:
df.head(10)

,Protocol,Packet Type,Traffic Type,Malware Indicators,Alerts/Warnings,Attack Type,Attack Signature,Action Taken,Severity Level,Network Segment,...,Source IP_Class,Destination IP_Class,Proxy_Class,Source Port_Cat,Destination Port_Cat,Packet Length_Quartile,Anomaly Scores_Quartile,Browser_family,OS_family,Device_type
0,ICMP,Data,HTTP,1,0,Malware,Known Pattern B,Logged,Low,Segment A,...,A,A,B,Registered,Registered,Q2: 421-782,Medium (25-49),IE,Windows,PC
1,ICMP,Data,HTTP,1,0,Malware,Known Pattern A,Blocked,Low,Segment B,...,A,A,noProxy,Registered,Registered,Q4: 1144-1500,High (50-74),IE,Windows,PC
2,UDP,Control,HTTP,1,1,DDoS,Known Pattern B,Ignored,Low,Segment C,...,A,C,A,Registered,Dynamic,Q1: 64-420,Very High (75-100),IE,Windows,PC
3,UDP,Data,HTTP,0,1,Malware,Known Pattern B,Blocked,Medium,Segment B,...,B,A,noProxy,Registered,Registered,Q1: 64-420,Low (0-24),Firefox,Mac OS X,PC
4,TCP,Data,DNS,0,1,DDoS,Known Pattern B,Blocked,Low,Segment C,...,A,B,B,Registered,Registered,Q4: 1144-1500,Low (0-24),IE,Windows,PC
5,UDP,Data,HTTP,0,0,Malware,Known Pattern A,Logged,Medium,Segment C,...,C,B,noProxy,Registered,Dynamic,Q4: 1144-1500,Low (0-24),Opera,Linux,PC
6,TCP,Data,DNS,0,0,DDoS,Known Pattern B,Ignored,High,Segment A,...,A,A,noProxy,Registered,Registered,Q1: 64-420,Medium (25-49),Opera,Linux,PC
7,ICMP,Data,DNS,1,1,Intrusion,Known Pattern A,Logged,High,Segment A,...,A,B,C,Registered,Registered,Q3: 783-1143,High (50-74),Chrome,Mac OS X,PC
8,TCP,Control,FTP,1,1,Intrusion,Known Pattern A,Blocked,High,Segment B,...,A,A,noProxy,Dynamic,Registered,Q4: 1144-1500,High (50-74),Chrome,Mac OS X,PC
9,UDP,Data,HTTP,0,1,Malware,Known Pattern B,Blocked,Medium,Segment A,...,A,B,A,Registered,Dynamic,Q1: 64-420,Low (0-24),Safari,Windows,PC


### Dataset1
Columns:
'Protocol', ==> non ordinal ==> 3 unique values
'Packet Type', ==> non ordinal ==> 2 unique values
'Traffic Type', ==> non ordinal ==> 3 unique values
'Malware Indicators', ==> non ordinal ==> 2 unique values
'Alerts/Warnings', ==> non ordinal ==> 2 unique values
'Attack Type', ==> TARGET
'Attack Signature', ==> non ordinal ==> 2 unique values
'Action Taken', ==> non ordinal ==> 3 unique values
'Severity Level', ==> ordinal ==> 3 unique values
'Network Segment', ==> non ordinal ==> 3 unique values
'Firewall Logs', ==> non ordinal ==> 2 unique values
'IDS/IPS Alerts', ==> non ordinal ==> 2 unique values
'Log Source', ==> non ordinal ==> 2 unique values
'is_weekend', ==> non ordinal ==> 2 unique values
'hours_div', ==> 6-20 and 21-5 ==> non ordinal ==> 2 unique values
'ContinentName_Source', ==> non ordinal ==> 8 unique values ==> Top 3 (90%) and Others
'ContinentName_Destination',==> non ordinal ==> 7 unique values ==> Top 3 (90%) and Others
'ContinentName_Proxy', ==> non ordinal ==> 7 unique values
'Source IP_Class', ==> non ordinal ==> 3 unique values
'Destination IP_Class',==> non ordinal ==> 3 unique values
'Proxy_Class', ==> non ordinal ==> 4 unique values
'Source Port_Cat', ==> non ordinal ==> 2 unique values
'Destination Port_Cat', ==> non ordinal ==> 2 unique values
'Packet Length_Quartile' 4 unique values ==> ordinal
'Anomaly Scores_Quartile', ==> 4 unique values ==> ordinal
'Browser_family', ==> non ordinal ==> 9 unique values
'OS_family', ==> non ordinal ==> 5 unique values
'Device_type' (4 unique values)

In [39]:
df['Anomaly Scores_Quartile'].value_counts()

Anomaly Scores_Quartile
High (50-74)          10119
Very High (75-100)    10011
Low (0-24)             9954
Medium (25-49)         9916
Name: count, dtype: int64

In [22]:
# Categorize the features to map the encoding types

# Define categories for ordinal features
ordinal_categories = {
    'Severity Level': ['Low', 'Medium', 'High'],
    'Packet Length_Quartile': ['Q1: 64-420', 'Q2: 421-782', 'Q3: 783-1143', 'Q4: 1144-1500'],
    'Anomaly Scores_Quartile': ['Low (0-24)', 'Medium (25-49)', 'High (50-74)', 'Very High (75-100)'],
}

# Non-ordinal features (binary) - use label encoding (k1_encoding)
binary_features = [
    'Packet Type',
    'Malware Indicators',
    'Alerts/Warnings',
    'Attack Signature',
    'Firewall Logs',
    'IDS/IPS Alerts',
    'Log Source',
    'is_weekend',
]

# Non-ordinal features (3+ categories) - use one-hot encoding
onehot_features = [
    'Protocol',
    'Traffic Type',
    'Action Taken',
    'Network Segment',
    'hours_div',
    'ContinentName_Source',
    'ContinentName_Destination',
    'ContinentName_Proxy',
    'Source IP_Class',
    'Destination IP_Class',
    'Proxy_Class',
    'Source Port_Cat',
    'Destination Port_Cat',
    'Browser_family',
    'OS_family',
    'Device_type',
]

# Apply ordinal encoding
print("\n[1] Applying Ordinal Encoding...")
for column, categories in ordinal_categories.items():
    if column in df.columns:
        df = pre.ordinal_encoding(df, column, categories_order=categories)
        print(f"    ✓ {column}")

# Apply K-1 encoding for binary features
print("\n[2] Applying K-1 Encoding (Binary Features)...")
for column in binary_features:
    if column in df.columns:
        df = pre.k1_encoding(df, column)
        print(f"    ✓ {column}")

# # Apply K-1 encoding for non-ordinal features with 3+ categories
print("\n[3] Applying K-1 Encoding (Multi-category Features)...")
for column in onehot_features:
    if column in df.columns:
        df = pre.k1_encoding(df, column)
        print(f"    ✓ {column}")

print("\n" + "="*60)
print("FEATURE ENGINEERING COMPLETED!")
print("="*60)
print(f"\nFinal dataset shape: {df.shape}")
print(f"Total features: {len(df.columns)}")


[1] Applying Ordinal Encoding...
    ✓ Severity Level
    ✓ Packet Length_Quartile
    ✓ Anomaly Scores_Quartile

[2] Applying K-1 Encoding (Binary Features)...
    ✓ Packet Type
    ✓ Malware Indicators
    ✓ Alerts/Warnings
    ✓ Attack Signature
    ✓ Firewall Logs
    ✓ IDS/IPS Alerts
    ✓ Log Source
    ✓ is_weekend

[3] Applying K-1 Encoding (Multi-category Features)...
    ✓ Protocol
    ✓ Traffic Type
    ✓ Action Taken
    ✓ Network Segment
    ✓ hours_div
    ✓ ContinentName_Source
    ✓ ContinentName_Destination
    ✓ ContinentName_Proxy
    ✓ Source IP_Class
    ✓ Destination IP_Class
    ✓ Proxy_Class
    ✓ Source Port_Cat
    ✓ Destination Port_Cat
    ✓ Browser_family
    ✓ OS_family
    ✓ Device_type

FEATURE ENGINEERING COMPLETED!

Final dataset shape: (40000, 51)
Total features: 51


In [23]:
df.columns

Index(['Protocol_TCP', 'Protocol_UDP', 'Packet Type_Data', 'Traffic Type_FTP',
       'Traffic Type_HTTP', 'Malware Indicators_1', 'Alerts/Warnings_1',
       'Attack Type', 'Attack Signature_Known Pattern B',
       'Action Taken_Ignored', 'Action Taken_Logged', 'Severity Level',
       'Network Segment_Segment B', 'Network Segment_Segment C',
       'Firewall Logs_1', 'IDS/IPS Alerts_1', 'Log Source_Server',
       'is_weekend_1', 'hours_div_6-20', 'ContinentName_Source_Europe',
       'ContinentName_Source_North America', 'ContinentName_Source_Other',
       'ContinentName_Destination_Europe',
       'ContinentName_Destination_North America',
       'ContinentName_Destination_Other', 'ContinentName_Proxy_North America',
       'ContinentName_Proxy_Other', 'ContinentName_Proxy_noProxy',
       'Source IP_Class_B', 'Source IP_Class_C', 'Destination IP_Class_B',
       'Destination IP_Class_C', 'Proxy_Class_B', 'Proxy_Class_C',
       'Proxy_Class_noProxy', 'Source Port_Cat_Registered'

In [24]:
# Save as CSV
df.to_csv('../data/processed_data/cybersecurity_attacks_dataset1_finalfeatures.csv', index=False)
print("DataFrame saved as CSV!")

DataFrame saved as CSV!
